In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "datasets").exists():
            return candidate
    raise RuntimeError("Repository root not found.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [1]:
import spacy
from spacy.cli import download
from transformers import pipeline
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment
import inspect
import time
import re
from difflib import SequenceMatcher

In [6]:
from src.core.retrieval import RetrievalEngine, find_substrings, find_all_matching_substrings

# Instancia a engine carregando o índice e o modelo
engine = RetrievalEngine(index_path="2025_sparse_wikititle_index", top_k=100, model="PL2")

JavaException: JVM exception occurred: Could not load an index for ref 2025_sparse_wikititle_index, even though IndexLoader org.terrier.structures.IndexOnDisk$DiskIndexLoader could support that type of index. It may be your ref had a wrong location; Terrier logs may have more information. java.lang.IllegalArgumentException

In [3]:
class NLU_Module:

    INTENT_THRESHOLD = 0.5
    SLOT_THRESHOLD = 0.5

    #   " \"{}\" represents the owner of playlist"
    #   " \"{}\" represents the artist name"

    #MoritzLaurer/ModernBERT-large-zeroshot-v2.0
    #MoritzLaurer/deberta-v3-large-zeroshot-v2.0

    def __init__(self, nli_model_name="MoritzLaurer/ModernBERT-large-zeroshot-v2.0", qa_model_name="bert-large-uncased-whole-word-masking-finetuned-squad", spacy_model="en_core_web_sm", device=0):
        self.nli_model_name = nli_model_name
        self.qa_model_name = qa_model_name
        self.spacy_model = spacy_model
        self.device = device
        self.classifier = self.load_classifier()
        self.qa_model = self.load_qa_model()
        self.nlp = self.load_spacy_model()
        print("\nClassifier and NLP model loaded successfully.\n")
        

    def load_intent_dict(self, intent_dict):
        self.intent_slot_dict = intent_dict
        
    def load_classifier(self):
        print(f"Loading zero-shot classifier: {self.nli_model_name}...")
        return pipeline("zero-shot-classification", model=self.nli_model_name, device=self.device)

    def load_qa_model(self):
        print(f"Loading question-answering model: {self.qa_model_name}...")
        return pipeline("question-answering", model=self.qa_model_name, device=self.device)

    def load_spacy_model(self):
        print(f"Loading spaCy model: {self.spacy_model}...")
        try:
            return spacy.load(self.spacy_model)
        except OSError:
            print(f"Modelo '{self.spacy_model}' não encontrado. Instalando...")
            download(self.spacy_model)
            return spacy.load(self.spacy_model)
    
    def intent_recognition(self, sentence):
        
        intents_list = list(self.intent_slot_dict.keys())
        results = self.classifier(
            sentence,
            intents_list,
            hypothesis_template="This command is intended to {}",
            multi_label=False
        )

        max_index = results["scores"].index(max(results["scores"]))
        top_score = results["scores"][max_index]

        if top_score >= self.INTENT_THRESHOLD:
            top_intent = results["labels"][max_index]
            return top_intent,round(top_score,3)

        return None
    

    '''def get_slot_tokens(self, sentence, slot_labels):
        doc = self.nlp(sentence)

        entities = [ent.text for ent in doc.ents]
        
        for token in doc:
            print(f"Token: {token.text}, POS: {token.pos_}, Dependência: {token.dep_}")
            
        
        nouns_and_tokens = []
        i = 0
        n = len(doc)
        
        while i < n:
            token = doc[i]
            # Verificação inicial: se o token está em entities, ou não tem o POS desejado, ou falha na verificação de slots
            if token.text in entities:
                i += 1
                continue
            if token.pos_ not in ("NOUN", "PROPN", "ADV", "PRON"):
                i += 1
                continue
            if not self.is_string_not_present(token.text):
                i += 1
                continue
    
            # Se não for NOUN nem PROPN, adiciona imediatamente
            if token.pos_ not in ("NOUN", "PROPN"):
                nouns_and_tokens.append(token.text)
                i += 1
                continue
    
            # Se for NOUN ou PROPN e não for compound, adiciona o token
            if token.dep_ != "compound":
                nouns_and_tokens.append(token.text)
                i += 1
            else:
                # Inicia a unificação de tokens compound
                start = i  # marca o início da sequência
                while i < n and doc[i].pos_ in ("NOUN", "PROPN"):
                    # Se chegar em um token que não é compound, incrementa e encerra o grupo
                    if doc[i].dep_ != "compound":
                        i += 1
                        break
                    i += 1
                
                # Recupera a substring exata do documento, preservando espaços e pontuação
                compound_phrase = doc[start:i].text
                
                # Remove substrings dos slot_labels (correspondência exata de palavra, case-insensitive)
                for slot_label in slot_labels:
                    # Itera sobre cada palavra do slot label
                    for word in slot_label.split():
                        pattern = re.compile(r"\b" + re.escape(word) + r"\b", re.IGNORECASE)
                        compound_phrase = pattern.sub("", compound_phrase)
                compound_phrase = compound_phrase.strip()
                if compound_phrase:
                    nouns_and_tokens.append(compound_phrase)

        print(nouns_and_tokens)

        tokens_list = []

        for token in nouns_and_tokens:
            for entitie in entities:
                if token.lower() in entitie.lower():
                    break
            else:
                tokens_list.append(token)
        
        return tokens_list'''

    ### NEW SINTATIC TOKEN EXTRACTION
    def get_slot_tokens(self, sentence, slot_labels):
        doc = self.nlp(sentence)

        # Captura as entidades já reconhecidas
        entities = [ent.text for ent in doc.ents]
        
        for token in doc:
            print(f"Token: {token.text}, POS: {token.pos_}, Dependência: {token.dep_}")
        
        nouns_and_tokens = []
        i = 0
        n = len(doc)
        
        # Incluímos VERB na lista de POS desejados para também capturar construções do tipo "cleaning the house"
        POS_DESIRED = ("NOUN", "PROPN", "ADV", "PRON", "VERB")
        
        while i < n:
            token = doc[i]
            # Se o token já for reconhecido como entidade, pulamos
            if token.text in entities:
                i += 1
                continue
            # Se o token não estiver nos POS desejados, pula
            if token.pos_ not in POS_DESIRED:
                i += 1
                continue
            # Se a sua verificação customizada falhar
            if not self.is_string_not_present(token.text):
                i += 1
                continue
            
            # Se o token for VERB e for a raiz (ROOT), ignoramos para não capturar toda a sentença
            if token.pos_ == "VERB" and token.dep_ == "ROOT":
                print(f"Ignorando VERB ROOT '{token.text}' para evitar capturar a sentença inteira.")
                i += 1
                continue

            # Se o token for VERB (e não for ROOT), tentamos capturar a subárvore (verbo + complementos)
            if token.pos_ == "VERB":
                subtree_tokens = list(token.subtree)
                if len(subtree_tokens) > 1:
                    start_sub = subtree_tokens[0].i
                    end_sub = subtree_tokens[-1].i + 1
                    phrase = doc[start_sub:end_sub].text
                    # Remove substrings dos slot_labels, se houver
                    for slot_label in slot_labels:
                        for word in slot_label.split():
                            pattern = re.compile(r"\b" + re.escape(word) + r"\b", re.IGNORECASE)
                            phrase = pattern.sub("", phrase)
                    phrase = phrase.strip()
                    if phrase:
                        print(f"Capturado VERB: '{phrase}' a partir da subárvore de '{token.text}'")
                        nouns_and_tokens.append(phrase)
                i += 1
                continue
            
            # Se for ADV ou PRON, adiciona imediatamente
            if token.pos_ in ("ADV", "PRON"):
                print(f"Capturado ADV/PRON: '{token.text}'")
                nouns_and_tokens.append(token.text)
                i += 1
                continue
            
            # Para NOUN e PROPN:
            # Se não for compound, adiciona o token isolado
            if token.dep_ != "compound":
                print(f"Capturado NOUN/PROPN (não compound): '{token.text}'")
                nouns_and_tokens.append(token.text)
                i += 1
            else:
                # Se for compound, unifica sequência de tokens
                start = i  # marca o início da sequência
                while i < n and doc[i].pos_ in ("NOUN", "PROPN"):
                    # Se chegar a um token que não é compound, encerra o grupo
                    if doc[i].dep_ != "compound":
                        i += 1
                        break
                    i += 1
                compound_phrase = doc[start:i].text
                # Remove substrings dos slot_labels (correspondência exata, case-insensitive)
                for slot_label in slot_labels:
                    for word in slot_label.split():
                        pattern = re.compile(r"\b" + re.escape(word) + r"\b", re.IGNORECASE)
                        compound_phrase = pattern.sub("", compound_phrase)
                compound_phrase = compound_phrase.strip()
                if compound_phrase:
                    print(f"Capturado compound: '{compound_phrase}'")
                    nouns_and_tokens.append(compound_phrase)
        
        print("nouns_and_tokens capturados:", nouns_and_tokens)
        
        tokens_list = []
        for token_str in nouns_and_tokens:
            for ent in entities:
                if token_str.lower() in ent.lower():
                    break
            else:
                tokens_list.append(token_str)
        return tokens_list
        

    def get_entities(self, sentence):
        doc = self.nlp(sentence)
        entities = [ent.text for ent in doc.ents]
        for ent in doc.ents:
            print(f"Texto: {ent.text} | Tipo: {ent.label_}")
        return entities
        
    def is_string_not_present(self, query):
        """
        Returns True if the query is not found as a substring in any intent or slot name.
        """
        query_lower = query.lower()
        for intent, data in self.intent_slot_dict.items():
            if query_lower in intent.lower():
                return False
            slots = data.get("slots", {})
            for slot_name in slots.keys():
                if query_lower in slot_name.lower():
                    return False
        return True

    def remove_duplicates(self, strings):
        """
        Remove duplicatas de uma lista de strings, comparando em lower case e mantendo a ordem.
        """
        unique = []
        seen = set()
        for s in strings:
            if s not in seen:
                unique.append(s)
                seen.add(s)
        return unique

    def remove_duplicates_and_substrings(self, strings):
        # Primeiro, remove duplicatas usando a função remove_duplicates.
        unique_strings = self.remove_duplicates(strings)
        
        # Em seguida, remove strings que são substrings de outra string na lista.
        filtered = []
        for s in unique_strings:
            # Se nenhuma outra string (em lower case) contiver 's' como substring (exceto quando s é exatamente igual),
            # então adiciona s à lista filtrada.
            if not any(s in other and s != other for other in unique_strings):
                filtered.append(s)
        return filtered

    

    def filter_doctitles(self, doc_titles, slot_labels):
        pos_discard = {"ADP", "ADV", "AUX", "INTJ", "CONJ", "CCONJ", "SCONJ", "VERB"}
    
        # 1. Remove títulos que são apenas uma palavra e que pertencem a pos_discard
        for i in reversed(range(len(doc_titles))):
            term = doc_titles[i]
            doc = self.nlp(term)
            if len(doc) == 1 and doc[0].pos_ in pos_discard:
                doc_titles.pop(i)
    
        # 2. Remover substrings que correspondam aos slot_labels (ignorar diferença de maiúsc./minúsc.)
        for i, title in enumerate(doc_titles):
            for slot_label in slot_labels:
                # Para cada palavra no slot label, remover correspondências exatas (word boundaries)
                for word in slot_label.split():
                    pattern = re.compile(r"\b" + re.escape(word) + r"\b", re.IGNORECASE)
                    title = pattern.sub("", title)
            # Remove espaços extras após as substituições
            doc_titles[i] = title.strip()
    
        return doc_titles
        
        
    def slot_scores(self, sentence, intent, tokens, validation_slots=None):
        if intent in self.intent_slot_dict:
            if not validation_slots:
                slots = self.intent_slot_dict[intent]["slots"]  # ou {} se preferir retornar vazio
            else:
                slots = {slot: info for slot, info in self.intent_slot_dict[intent]["slots"].items() if slot in validation_slots}
            
        if not slots:
            return slots
    
        slot_token_scores = {}
        for slot_name, slot_info in slots.items():
            slot_token_scores[slot_name] = []
            output = self.classifier(
                sentence,
                tokens,
                hypothesis_template=slot_info["hypothesis"],
                multi_label=True
            )

            print(f"\n Output for hypothesis '{slot_info["hypothesis"]}':\n{output}\n")

            for label, score in zip(output["labels"], output["scores"]):
                slot_token_scores[slot_name].append({
                    "token": label,
                    "score": score
                })
            
            #print(f'\n {slot_token_scores}')
        
        data_list = [(slot, item['token'], item['score']) for slot, items in slot_token_scores.items() for item in items]

        # Criar um DataFrame a partir da lista
        df = pd.DataFrame(data_list, columns=['Slot', 'Token', 'Score'])

        # Transformar para formato de matriz com 'Slot' como índice e 'Token' como colunas
        df_score = df.pivot(index='Slot', columns='Token', values='Score')

        #print(f'\n {df_score}')

        #dummy_col_name = "NULL"

        # Adicionar a coluna dummy diretamente ao DataFrame
        #df_score[dummy_col_name] = self.SLOT_THRESHOLD  # O threshold será invertido depois

        print(f'\n {df_score}')

        
        return df_score
    '''
    def slot_filling(self, df_score):

        # Transformar em matriz de custo (multiplicar por -1)
        cost_matrix = -df_score.to_numpy()

        # Hungarian algorithm
        row_ind, col_ind = linear_sum_assignment(cost_matrix)

        # 4) Interpreta resultados
        assigned_pairs = []
        for i, j in zip(row_ind, col_ind):
            slot_name = df_score.index[i]
            token_name = df_score.columns[j]
            
            # Se foi atribuído ao "NULL", significa que o algoritmo preferiu não escolher um token real
            #if token_name == df_score.columns[-1]:
                # Pulamos essa atribuição
                #continue
            
            # Recupera o score original (score = -cost)
            score = df_score.iloc[i, j]
            
            assigned_pairs.append((slot_name, token_name, score))
            
        slot_filling = {}

        for slot, token, score in assigned_pairs:
            slot_filling[slot] = (token,round(float(score),3))

            # Verifica quais slots não foram atribuídos (faltando no dicionário)
        for slot in df_score.index:
            if slot not in slot_filling:
                slot_filling[slot] = ("NULL", 0.0)
            
        return slot_filling
        '''
    
    def slot_filling(self, df_score):
        threshold = 0.2
        huge_cost = 1e6  # valor alto para desestimular associações com score abaixo do limiar
    
        # Converte o DataFrame para uma matriz de scores
        scores = df_score.to_numpy()
    
        # Constrói a matriz de custos:
        # Se score >= threshold, custo = -score (para maximizar o score);
        # se score < threshold, custo = huge_cost (para tornar essa associação indesejável).
        cost_matrix = np.where(scores >= threshold, -scores, huge_cost)
    
        # Aplica o algoritmo húngaro para minimizar o custo total
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
        # Interpreta os resultados
        assigned_pairs = []
        for i, j in zip(row_ind, col_ind):
            slot_name = df_score.index[i]
            token_name = df_score.columns[j]
            score = df_score.iloc[i, j]  # score original
            assigned_pairs.append((slot_name, token_name, score))
    
        # Cria o dicionário de associações, substituindo por "NULL" quando o score não atingir o limiar
        slot_filling = {}
        for slot, token, score in assigned_pairs:
            if score < threshold:
                slot_filling[slot] = ("NULL", 0.0)
            else:
                slot_filling[slot] = (token, round(float(score), 3))
    
        # Garante que todos os slots tenham uma entrada, atribuindo "NULL" aos não associados
        for slot in df_score.index:
            if slot not in slot_filling:
                slot_filling[slot] = ("NULL", 0.0)
    
        return slot_filling

    
    def validate_slot_assignments(self, sentence, result):
        intent = result['intent'][0]
        validated_slots = {}
        
        # Obtém os slots definidos para o intent a partir do INTENT_SLOT_DICT
        intent_slots = self.intent_slot_dict.get(intent, {}).get("slots", {})
        if not intent_slots:
            print(f"Nenhum slot definido para o intent '{intent}'.")
            return {"intent": result["intent"], "slots": {}}
        
        # Os rótulos candidatos são os slots definidos para o intent, adicionando "other"
        #candidate_labels = list(intent_slots.keys()) + ["unknown"]
        
        # Itera sobre cada par slot-token do resultado
        for slot, (token, score) in result['slots'].items():
            # Se o token for 'NULL', mantém a associação sem validação
            if token == 'NULL' or score < 0.1:
                validated_slots[slot] = ('NULL', 0.0, 0.0)
                continue
            
            # Se o score for maior que 0.7, não é analisado; apenas mantém o valor original.
            if score > 0.7:
                validated_slots[slot] = (token, score)
                continue
                
            # Define a hipótese usando o token e o intent.
            # Exemplo: "I want to find a restaurant, and {token} is the {}"
            #hypothesis = f"I want to {intent.lower()}, and \"{token}\" represents the {{}}"
            hypothesis = f"\"{token}\" represents the {{}}"
            candidate_labels =[slot, "other"]
            
            # Aplica o zero-shot classifier para testar a hipótese
            classification_result = self.classifier(
                sentence,
                candidate_labels,
                hypothesis_template=hypothesis,
                multi_label=False
            )
            
            print(hypothesis)
            print(classification_result)
            
            # Recupera o score correspondente ao label esperado (slot)
            try:
                index_label = classification_result['labels'].index(slot)
                predicted_score = classification_result['scores'][index_label]
            except ValueError:
                predicted_score = 0.0
            
            if self._is_valid_assignment(expected_score=predicted_score, scores=classification_result['scores']):
                validated_slots[slot] = (token, score)
            else:
                validated_slots[slot] = ("NULL", 0.0)
                print(f"Validation failed for slot '{slot}' with token '{token}'. "
                      f"Predicted expected score: {predicted_score:.3f} is not dominant among {classification_result['scores']}.")
        
        # Retorna o resultado com os slots validados, mantendo também os que não foram analisados
        return {"intent": result["intent"], "slots": validated_slots}


    '''def get_intent_slot_answers(self, intent, sentence):
        """
        Para o intent informado, percorre os slots e extrai as respostas utilizando o QA pipeline.
        As respostas de cada slot são agrupadas num dicionário, onde cada chave é o nome do slot e o
        valor é uma lista de respostas (cada resposta é um dict contendo 'answer', 'score', 'start' e 'end').
        
        Parâmetros:
            - context: o contexto de onde extrair as respostas.
            - intent: o intent a ser processado (chave do dicionário de intents).
            - qa_pipeline: pipeline de QA que receberá a pergunta e o contexto.
            - top_k: número de respostas a serem retornadas para cada slot.
            
        Retorna:
            - Um dicionário com as respostas agrupadas por slot.
        """
        # Carrega o dicionário de intents e slots
        #intent_slot_dict = self.load_intent_dict()
           
        # Recupera os slots do intent
        slots = self.intent_slot_dict[intent]["slots"]
        slot_answers = {}
        
        
        # Para cada slot, utiliza a "question" específica para extrair respostas com o pipeline
        for slot_label, slot_info in slots.items():
            slot_answers[slot_label] = []
            # A pergunta vem do campo "question" de cada slot
            question = slot_info["question"]
            
            # Executa o pipeline de QA com o parâmetro top_k (que retorna uma lista de respostas)
            results = self.qa_model(question=question, context=sentence)
            
            # Armazena os resultados para o slot
            slot_answers[slot_label] = results

        print(slot_answers)

        unique_answers = []
        for slot, answer_info in slot_answers.items():
            answer = answer_info['answer']
            if answer not in unique_answers:
                unique_answers.append(answer)
        
        print("Respostas únicas:", unique_answers)
        
        return unique_answers'''

    ### NOVA QUESTION ANSWER ELIMINANDO REDUNDANCIA

    def get_intent_slot_answers(self, intent, sentence):
        """
        Para o intent informado, percorre os slots e extrai as respostas utilizando o QA pipeline.
        As respostas de cada slot são agrupadas num dicionário, onde cada chave é o nome do slot e o
        valor é uma lista de respostas (cada resposta é um dict contendo 'answer', 'score', 'start' e 'end').
        
        Em seguida, as respostas são combinadas e ordenadas pelo score (maior primeiro). A partir da resposta 
        com maior score, eliminamos as respostas que contiverem essa resposta ou que estejam contidas nela, 
        de modo a evitar redundância.
    
        Parâmetros:
          - intent: o intent a ser processado (chave do dicionário de intents).
          - sentence: a sentença (contexto) de onde extrair as respostas.
        
        Retorna:
          - Uma lista de respostas únicas (strings) após a remoção de redundâncias.
        """
        # 1. Recupera os slots para o intent
        slots = self.intent_slot_dict[intent]["slots"]
        slot_answers = {}
    
        # 2. Para cada slot, utiliza a pergunta específica para extrair respostas via QA pipeline
        for slot_label, slot_info in slots.items():
            slot_answers[slot_label] = []
            question = slot_info["question"]
    
            # Executa o QA pipeline; se o resultado for um dicionário, converte para lista
            results = self.qa_model(question=question, context=sentence)
            if isinstance(results, dict):
                results = [results]
            slot_answers[slot_label] = results
    
        print("Respostas extraídas por slot:")
        for slot, answers in slot_answers.items():
            print(f"  Slot '{slot}':")
            for ans in answers:
                print(f"    Answer: {ans['answer']}, Score: {ans['score']}")
    
        # 3. Combina todas as respostas em uma lista de tuplas: (slot, answer, score)
        items = []
        for slot_label, answers in slot_answers.items():
            for ans in answers:
                items.append((slot_label, ans['answer'], ans['score']))
    
        # 4. Ordena as respostas pelo score em ordem decrescente (maior score primeiro)
        items_sorted = sorted(items, key=lambda x: x[2], reverse=True)
        print("\nRespostas ordenadas por score:")
        for slot, answer, score in items_sorted:
            print(f"  Slot: {slot}, Answer: '{answer}', Score: {score}")
    
        # 5. Elimina redundâncias: para cada resposta de maior score, descarta as que contiverem ela ou que
        # estejam contidas nela.
        final_candidates = []
        for candidate in items_sorted:
            candidate_text = candidate[1].lower()
            redundant = False
            # Compara com as respostas já selecionadas (de score maior)
            for final_candidate in final_candidates:
                final_text = final_candidate[1].lower()
                # Se uma resposta estiver contida na outra, a candidata é redundante
                if candidate_text in final_text or final_text in candidate_text:
                    print(f"Ignorando redundância: '{candidate[1]}' (comparado com '{final_candidate[1]}')")
                    redundant = True
                    break
            if not redundant:
                print(f"Adicionando resposta: '{candidate[1]}'")
                final_candidates.append(candidate)
    
        # 6. Extrai apenas os textos únicos das respostas finais
        unique_answers = []
        for _, answer, _ in final_candidates:
            if answer not in unique_answers:
                unique_answers.append(answer)
        
        print("\nRespostas únicas finais:", unique_answers)
        return unique_answers

    



    def _is_valid_assignment(self, expected_score, scores, z_threshold=0):
        """
        Verifica se o score do label esperado é dominante na distribuição.
        
        Calcula:
          - ratio: expected_score / top_score
          - z_score: (expected_score - mean) / std (da distribuição dos scores)
          
        Retorna True se ambos os valores forem maiores ou iguais aos limiares definidos.
        """
        scores_array = np.array(scores)
        top_score = np.max(scores_array)
        
        mean_score = np.mean(scores_array)
        std_score = np.std(scores_array)
        z_score = (expected_score - mean_score) / std_score if std_score > 0 else 0.0
        
        # Debug: mostrar os valores calculados (pode ser comentado depois)
        print(f"Expected score: {expected_score:.3f}, Top score: {top_score:.3f}, Mean: {mean_score:.3f}, Std: {std_score:.3f}, Z-score: {z_score:.3f}")
        
        return z_score >= z_threshold


    def check_slots(self, sequence, intent):
        # Obtém a lista de slots para o intent informado
        slots = list(self.intent_slot_dict[intent]["slots"].keys())
        
        # Define o template da hipótese usando o slot (candidate label)
        hypothesis = "The sentence mentions the {}"
        
        # Aplica o zero-shot classifier para testar a hipótese para todos os slots de uma vez
        classification_result = self.classifier(
            sequence,
            slots,
            hypothesis_template=hypothesis,
            multi_label=True
        )
        
        # Cria um dicionário para armazenar os slots cujo score seja maior que 0.5.
        # Supomos que os rótulos (labels) retornados estão na mesma ordem dos candidate labels (slots)
        slot_results = {}
        for slot, score in zip(classification_result["labels"], classification_result["scores"]):
            slot_results[slot] = score
    
        return slot_results

        
        
        
        
        
        


In [4]:
class TimeTracker:
    def __init__(self):
        self.start_time = None
        self.times = []
        self.global_start_time = None  # Tempo total desde o início

    def start(self):
        """Inicia a contagem do tempo."""
        self.start_time = time.time()
        self.global_start_time = self.start_time  # Marca o tempo total
        self.times = []  # Reinicia o histórico

    def mark(self, label):
        """Marca o tempo atual e reinicia a contagem."""
        if self.start_time is None:
            raise RuntimeError("O TimeTracker precisa ser iniciado primeiro com start()")
        
        elapsed_time = time.time() - self.start_time
        self.times.append((label, elapsed_time))
        print(f"{label}: {elapsed_time:.6f} segundos")
        
        # Reinicia a contagem
        self.start_time = time.time()

    def total_time(self):
        """Retorna o tempo total desde o início."""
        if self.global_start_time is None:
            return None
        return time.time() - self.global_start_time

    def results(self):
        """Retorna todas as medições registradas."""
        return self.times

In [5]:
tracker = TimeTracker()
nlu_module = NLU_Module()

Loading zero-shot classifier: MoritzLaurer/ModernBERT-large-zeroshot-v2.0...


Device set to use cuda:0


Loading question-answering model: bert-large-uncased-whole-word-masking-finetuned-squad...


Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Loading spaCy model: en_core_web_sm...

Classifier and NLP model loaded successfully.



In [6]:
INTENT_SLOT_DICT = {
    "Book a flight": {
        "slots": {
            "departure": {
                "hypothesis": "The departure city of the flight is {}",
                "question": "What is the departure city of the flight mentioned in the text?"
            },
            "destination": {
                "hypothesis": "The destination city of the flight is {}",
                "question": "What is the destination city of the flight mentioned in the text?"
            },
            "class": {
                "hypothesis": "The class of the flight is {}",
                "question": "What is the class of the flight mentioned in the text?"
            }
        },
    },
    "Book a train": {
        "slots": {
            "departure station": {
                "hypothesis": "The departure station for the train is {}",
                "question": "What is the departure station for the train mentioned in the text?"
            },
            "arrival station": {
                "hypothesis": "The destination station for the train is {}",
                "question": "What is the arrival station for the train mentioned in the text?"
            },
            "class": {
                "hypothesis": "The class of the train is {}",
                "question": "What is the class of the train mentioned in the text?"
            }
        },
    },
    "Book a hotel room": {
        "slots": {
            "city": {
                "hypothesis": "The city for the hotel booking is {}",
                "question": "What is the city for the hotel booking mentioned in the text?"
            },
            "number of guests": {
                "hypothesis": "The number of guests for the hotel booking is {}",
                "question": "What is the number of guests for the hotel booking mentioned in the text?"
            },
            "room type": {
                "hypothesis": "The room type for the hotel booking is {}",
                "question": "What is the room type for the hotel booking mentioned in the text?"
            }
        },
    },
    "Check flight status": {
        "slots": {
            "flight number": {
                "hypothesis": "The flight number to check the status is {}",
                "question": "What is the flight number to check the status mentioned in the text?"
            },
            "departure date": {
                "hypothesis": "The departure date of the flight to check the status is {}",
                "question": "What is the departure date of the flight to check the status mentioned in the text?"
            }
        },
    },
    "Find a restaurant": {
        "slots": {
            "location": {
                "hypothesis": "The location of the restaurant is \"{}\"",
                "question": "What is the location to find a restaurant mentioned in the text?"
            },
            "cuisine type": {
                "hypothesis": "The type of cuisine is \"{}\"",
                "question": "What is the type of cuisine for the restaurant mentioned in the text?"
            },
            "number of people": {
                "hypothesis": "The number of people for the restaurant reservation is \"{}\"",
                "question": "What is the number of people for the restaurant reservation mentioned in the text?"
            },
            "restaurant name": {
                "hypothesis": "The name of the restaurant is \"{}\"",
                "question": "What is the name of the restaurant mentioned in the text?"
            },
            "reservation time": {
                "hypothesis": "The time for reservation is \"{}\"",
                "question": "What is the time range mentioned in the text?"
            }
        },
    },
    "Cancel a flight": {
        "slots": {
            "booking reference": {
                "hypothesis": "The booking reference of the flight to cancel is {}",
                "question": "What is the booking reference of the flight to cancel mentioned in the text?"
            },
            "last name": {
                "hypothesis": "The last name of the passenger for the flight cancellation is {}",
                "question": "What is the last name of the passenger for the flight cancellation mentioned in the text?"
            }
        },
    },
    "Check weather": {
        "slots": {
            "location": {
                "hypothesis": "The location to check the weather is \"{}\"",
                "question": "What is the location to check the weather mentioned in the text?"
            },
            "date": {
                "hypothesis": "The date to check the weather is \"{}\"",
                "question": "What is the date to check the weather mentioned in the text?"
            }
        },
    },
    "Play a game": {
        "slots": {
            "game": {
                "hypothesis": "The name of the game the user wants to play is {}",
                "question": "What is the name of the game the user wants to play mentioned in the text?"
            },
            "map": {
                "hypothesis": "The name of the map or location the user wants to play on is {}",
                "question": "What is the name of the map or location the user wants to play on mentioned in the text?"
            },
            "character": {
                "hypothesis": "The name of the character the user wants to play as is {}",
                "question": "What is the name of the character the user wants to play as mentioned in the text?"
            }
        }
    },
    "Watch a movie": {
        "slots": {
            "movie name": {
                "hypothesis": "The name of the movie the user wants to watch is {}",
                "question": "What is the name of the movie the user wants to watch mentioned in the text?"
            }
        }
    },
    "Add to a playlist": {
        "filter": "playlist",
        "slots": {
            "playable content": {
                "hypothesis": "The playable content is \"{}\"",
                "question": "What is the audio content mentioned in the text?"
            },
            "playlist title": {
                "hypothesis": "The title of the playlist is \"{}\"",
                "question": "What is the playlist title mentioned in the text?"
            },
            "playlist owner": {
                "hypothesis": "The owner of the playlist is \"{}\"",
                "question": "Who is the owner of the playlist mentioned in the text?"
            },
            "artist name": {
                "hypothesis": "The name of the artist is \"{}\"",
                "question": "Which artist or band is mentioned in the text?"
            }
        }
    },
    "Rate an item": {
    "slots": {
      "object name": {
        "hypothesis": "The name of the item to be rated is \"{}\"",
        "question": "What is the name of the item to be rated that is mentioned in the text?"
      },
      "object select": {
        "hypothesis": "The pronoun used to identify the item is \"{}\"",
        "question": "What is the reference or pronoun the user used to identify the item in the text?"
      },
      "object type": {
        "hypothesis": "The item is part of a \"{}\"",
        "question": "What is the type of item being rated mentioned in the text?"
      },
      "object part of series type": {
        "hypothesis": "The item type is \"{}\"",
        "question": "Is the item part of a series or collection mentioned in the text? If so, what type?"
      },
      "rating value": {
        "hypothesis": "The rating number is \"{}\"",
        "question": "What is the rating value the user assigned to the item?"
      },
      "best rating": {
        "hypothesis": "The maximum of rating scale the user mentions is \"{}\"",
        "question": "What is the maximum rating scale (the best rating) mentioned in the text?"
      },
      "rating unit": {
        "hypothesis": "The rating metric to the item is \"{}\"",
        "question": "Which unit (stars, points, etc.) is the user using to rate the item?"
      }
    }
  }
}


nlu_module.load_intent_dict(INTENT_SLOT_DICT)

In [7]:
def normalize_text(text):
    # Extrai somente palavras (letras, dígitos) ignorando pontuações
    # Converte para lower-case e junta as palavras novamente
    words = re.findall(r'\w+', text)
    return words


def non_max_suppression_tokens_and_slots(hungarian_assignment, candidate_tokens, threshold_diff=0.0):
    """
    Realiza uma supressão não-máxima considerando o resultado do Hungarian algorithm.
    Para cada trio (slot, token, score) obtido, em ordem decrescente de score, remove da lista 
    de tokens candidatos os tokens que compartilham palavras (case-insensitive) com o token de referência.
    
    Além disso, ao usar um token como referência, esse token é removido da lista de candidatos
    e o slot associado é considerado "final" (ou seja, removido da lista de slots para análises futuras).
    
    Parâmetros:
        hungarian_assignment (dict): dicionário no formato
            { slot: (token_str, score), ... } ou com a chave "slots" contendo esse dicionário.
        candidate_tokens (list): lista de tokens que fazem parte da matriz de scores.
        threshold_diff (float): (não utilizado aqui, mas pode ser empregado para definir um limiar de remoção)
    
    Retorna:
        tuple: (final_slots, remaining_tokens, filtered_assignment)
            - final_slots: lista dos slots finais (usados como referência e não suprimidos por outros)
            - remaining_tokens: lista dos tokens candidatos restantes (não removidos)
            - filtered_assignment: hungarian assignment filtrado, contendo apenas os trios dos slots em final_slots.
    """
    # Se o dicionário possui a chave "slots", extrai apenas esse conteúdo.
    if "slots" in hungarian_assignment:
        hungarian_assignment = hungarian_assignment["slots"]
    
    # Converter o hungarian_assignment em uma lista de tuplas: (slot, token, score)
    items = [(slot, token, score) for slot, (token, score) in hungarian_assignment.items()]

    # Extrair apenas os slots para uma nova lista
    slot_list = [slot for slot, _, _ in items]
    
    # Ordenar os trios por score decrescente; em caso de empate, pelo comprimento do token
    items.sort(key=lambda x: (x[2], len(x[1])), reverse=True)
    
    # Conjuntos para tokens e slots removidos
    removed_tokens = set()
    removed_slots = set()
    
    # Lista que será atualizada com os tokens candidatos restantes (iniciada com todos)
    updated_candidates = candidate_tokens.copy()
    
    # Lista dos slots finais (aqueles que foram processados como referência)
    final_slots = []
    
    # Iterar pelos trios ordenados
    for slot, ref_token, ref_score in items:
        # Se esse token ou slot já foi removido, pula
        if ref_token in removed_tokens or slot in removed_slots:
            continue
        
        # Estamos usando esse token como referência, portanto o marcamos como removido
        removed_tokens.add(ref_token)
        final_slots.append(slot)
        
        # Tokeniza o token de referência (conjunto de palavras em minúsculas)
        #ref_words = set(ref_token.lower().split())
        ref_words = set(normalize_text(ref_token))
        
        
        # Itera sobre os tokens candidatos (fazendo uma cópia para permitir remoções durante a iteração)
        for cand in updated_candidates.copy():
            # Evita comparar o token de referência com ele mesmo
            if cand == ref_token:
                continue
            
            # Tokeniza o candidato
            #cand_words = set(cand.lower().split())
            cand_words = set(normalize_text(cand))
            
            # Se há interseção de palavras entre o token de referência e o candidato, remove o candidato
            print("ref words:", ref_words, "\ncand_words:", cand_words, "\n cand:", cand)
            if ref_words & cand_words:
                removed_tokens.add(cand)
                updated_candidates.remove(cand)
                # Remove também o slot associado a esse token candidato:
                for s, t, sc in items:
                    if t == cand:
                        removed_slots.add(s)
                        break
    
    # Filtra o hungarian_assignment mantendo apenas os slots que estão em final_slots
    filtered_assignment = {slot: hungarian_assignment[slot] for slot in hungarian_assignment if slot in final_slots}
    # Retorna os slots removidos (os que foram suprimidos), seguidos dos slots finais, os tokens restantes e o assignment filtrado
    #print(list(removed_slots)+final_slots)
    print(slot_list)
    print(updated_candidates)
    print(filtered_assignment)
    return slot_list, updated_candidates, filtered_assignment


In [8]:
def select_max_triples(df):
    """
    Recebe um DataFrame com linhas e colunas cujos elementos são scores (0 a 1) e retorna
    uma lista de trios (linha, coluna, score) selecionados de forma a que cada linha e cada
    coluna apareçam apenas uma vez, escolhendo o maior score para cada par.
    """
    # 1. Gerar a lista de trios
    triples = []
    for row in df.index:
        for col in df.columns:
            score = df.loc[row, col]
            triples.append((row, col, score))
    
    # 2. Ordenar os trios pelo score, do maior para o menor
    triples.sort(key=lambda x: x[2], reverse=True)
    
    # 3. Selecionar os trios, removendo aqueles que compartilham linha ou coluna
    selected = []
    used_rows = set()
    used_cols = set()
    
    for row, col, score in triples:
        # Se a linha ou a coluna já foi usada, pulamos esse trio
        if row in used_rows or col in used_cols:
            continue
        # Caso contrário, selecionamos esse trio e marcamos a linha e a coluna como usadas
        selected.append((row, col, score))
        used_rows.add(row)
        used_cols.add(col)
    
    return selected

def rescore_dataframe(df, score_dict):
    """
    Multiplica cada valor de cada linha do dataframe pelo fator correspondente
    presente em score_dict (onde as chaves são os slots).
    
    Args:
        df (pd.DataFrame): DataFrame com os scores dos tokens, onde o índice são os slots.
        score_dict (dict): Dicionário com os slots e seus fatores de re-score.
        
    Returns:
        pd.DataFrame: DataFrame com os scores atualizados.
    """
    # Converte o dicionário em uma Series, onde os índices coincidem com os do dataframe
    multipliers = pd.Series(score_dict)
    
    # Multiplica cada linha do dataframe pelo fator correspondente
    df_rescored = df.multiply(multipliers, axis=0)
    
    return df_rescored

In [9]:
def main(sentence):

    #start time tracker
    tracker.start()
    intent, intent_score = nlu_module.intent_recognition(sentence)
    tracker.mark("Intent reconhecido")

    if intent is None:
        print("Intent not recognized. Exiting...")
        return None

    slot_labels = list(nlu_module.intent_slot_dict[intent].get("slots", {}).keys())

    
    syntactic_slots = nlu_module.get_slot_tokens(sentence, slot_labels)
    print(f"Syntactic_slots: \n {syntactic_slots}")


    
    entities = nlu_module.get_entities(sentence)
    print(f"Entities: \n {entities}")
    


    
    query_results = engine.retrieve(sentence)
    doc_titles = query_results['title'].tolist()
    print(doc_titles)


    
    slots_retrieved = find_all_matching_substrings(doc_titles, sentence)
    slots_retrieved = nlu_module.filter_doctitles(slots_retrieved, slot_labels)
    print(f"Slots retrieved: \n {slots_retrieved}")



    
    relevant_tokens = nlu_module.remove_duplicates_and_substrings(syntactic_slots + slots_retrieved)
    print(f"Relevant tokens: \n {relevant_tokens}")
    tracker.mark("Tokens extraídos")


    qa_tokens = nlu_module.get_intent_slot_answers(intent, sentence)


    final_slot_tokens = nlu_module.remove_duplicates(relevant_tokens + entities + qa_tokens)
    final_slot_tokens = final_slot_tokens
    print(f"Final slot tokens: \n {final_slot_tokens}")



    
    df_slot_score = nlu_module.slot_scores(sentence, intent, final_slot_tokens)
    df_backup = df_slot_score
    tracker.mark("Matriz de score slot-token concluida")

    # VALIDAÇÃO E ATRIBUIÇÃO


    # NORMAL HUNGARIAN

    print("\n\n NORMAL HUNGARIAN \n\n ###################################\n\n")

    slots = nlu_module.slot_filling(df_slot_score)
    tracker.mark("Hungarian algoritm concluido")

    result = {'intent': (intent, intent_score), 'slots': slots}
    print(result)


    # REESCORED NORMAL HUNGARIAN

    print("\n\n REESCORED NORMAL HUNGARIAN \n\n ###################################\n\n")


    #reescore dataframe
    slots_checked = nlu_module.check_slots(sentence, intent)
    print("slots check score: ",slots_checked)
    df_slot_score = rescore_dataframe(df_slot_score, slots_checked)
    df_slot_score_scored = df_slot_score

    print(df_slot_score_scored)

    slots_scored = nlu_module.slot_filling(df_slot_score_scored)
    result_scored = {'intent': (intent, intent_score), 'slots': slots_scored}
    print(result_scored)



    # HUNGARIAN SEM REDUNDANCIA DE TOKENS

    print("\n\n HUNGARIAN SEM REDUNDANCIA \n\n ###################################\n\n")


    
    removed_slots, remaining_tokens, filtered_assignment = non_max_suppression_tokens_and_slots(result,final_slot_tokens)
    
    df_validation_slot = nlu_module.slot_scores(sentence, intent, remaining_tokens, removed_slots)
    validated_slots = nlu_module.slot_filling(df_validation_slot)

    combined_slots = {**filtered_assignment, **validated_slots}

    tracker.mark("Validação concluida")

    validated_results = {'intent': (intent, intent_score), 'slots': combined_slots}
    print(validated_results)
    

    
    # HUNGARIAN REESCORED SEM REDUNDANCIA DE TOKENS


    print("\n\n HUNGARIAN REESCORED SEM REDUNDANCIA DE TOKENS \n\n ###################################\n\n")


    removed_slots, remaining_tokens, filtered_assignment = non_max_suppression_tokens_and_slots(result,final_slot_tokens)
    
    df_validation_slot_pre_scored = nlu_module.slot_scores(sentence, intent, remaining_tokens, removed_slots)
    df_validation_slot_scored = rescore_dataframe(df_validation_slot_pre_scored, slots_checked)
    validated_slots = nlu_module.slot_filling(df_validation_slot_scored)

    combined_slots = {**filtered_assignment, **validated_slots}

    tracker.mark("Validação concluida")

    validated_results = {'intent': (intent, intent_score), 'slots': combined_slots}
    print(validated_results)


    
    tracker.mark("Hungarian algoritm concluido")


    # MAX ASSOCIATIONS SEM REDUNDANCIA DE TOKENS PÓS HUNGARIAN

    print("\n\n MAX ASSOCIATIONS SEM REDUNDANCIA DE TOKENS PÓS HUNGARIAN \n\n ###################################\n\n")


    new_validation = select_max_triples(df_validation_slot)
    print(new_validation)

    
    # MAX ASSOCIATIONS SCORED SEM REDUNDANCIA DE TOKENS PÓS HUNGARIAN

    print("\n\n MAX ASSOCIATIONS SCORED SEM REDUNDANCIA DE TOKENS PÓS HUNGARIAN \n\n ###################################\n\n")

    
    new_validation = select_max_triples(df_validation_slot_scored)
    print(new_validation)


    '''
    removed_slots, remaining_tokens, filtered_assignment = non_max_suppression_tokens_and_slots(result,final_slot_tokens)


    
    df_validation_slot_score = nlu_module.slot_scores(sentence, intent, remaining_tokens, removed_slots)
    validated_slots = nlu_module.slot_filling(df_validation_slot_score)

    combined_slots = {**filtered_assignment, **validated_slots}

    tracker.mark("Validação concluida")

    validated_results = {'intent': (intent, intent_score), 'slots': combined_slots}
    print(validated_results)
    
    print("\nResumo das medições:")
    for label, elapsed in tracker.results():
        print(f"{label}: {elapsed:.6f} segundos")

    # Exibir tempo total
    print(f"\nTempo total decorrido: {tracker.total_time():.6f} segundos")


    new_validation = select_max_triples(df_slot_score_scored)
    print(new_validation)


    print(nlu_module.check_slots(sentence, intent))

    '''
    return df_backup


    

In [ ]:
if __name__ == "__main__":
    
    #
    #sentence = "I need to see mother joan of the angels in one second"
    #sentence = "I'm trying to book a flight for tomorrow from New York to China"
    #sentence = "Please book a flight for me with departure city Boston, destination city Paris, and class business; also note my departure time is 10AM."
    #sentence = "Please book a flight for me with departure city Boston, destination city Paris"
    #sentence = "I need to book a train ticket from King's Cross Station to Victoria Station in first class, and please reserve a window seat as well."
    #sentence = "I want to book a hotel room in San Francisco for 3 guests in a suite, and also set my checkin date to August 15th with a breakfast option."
    #sentence = "Could you check flight status for flight AA1234 departing on 2025-03-12, and also update the passenger name to Michael?"
    #sentence = "Help me find a restaurant in Downtown Manhattan serving Mediterranean cuisine for 4 people at Le Petit Chef, and please note the reservation time is 7:30 PM."
    #sentence = "I need to cancel my flight with booking reference BRX456789 under the last name O'Connor, and also mention the reason is a schedule change."
    #sentence = "Please check day weather in Seattle on 2025-02-28, and also note that the time of day is morning in Seattle."
    #sentence = "I want to play a game called League of Legends on the Summoner's Rift map as Ahri, and also set the difficulty to hard mode."
    #sentence = "I'm planning to watch a movie titled Inception, and please adjust the screen size to large screen."
    #sentence = "Help me find a restaurant in downtown New York serving French cuisine for 2 people at Le Bernardin, and please note my dietary preference for gluten-free options."
    #sentence = "Help me find a restaurant in downtown São Paulo serving Brazilian cuisine for 4 people at Fogo de Chão, and please note the reservation time is 8:00 PM."
    #sentence = "Help me find a restaurant in downtown New York serving American cuisine for 4 people at The Capital Grill, ensuring it's not confused with Capital Grille, and please note the reservation time is 7:00 PM."
    #sentence = "Add clem burke in my playlist Pre-Party R&B Jams"
    #sentence = "Add the album to my flow español playlist"
    #sentence = "add this song by Too Poetic to my Piano Ballads playlist"
    #sentence = "Add this album to Old School Death Metal"
    #sentence = "Add the artist with name nuba to my Metal Party playlist"
    #sentence = "Put jean philippe goncalves onto my running to rock 170 to 190 bpm playlist."
    #sentence = "Add Crying, Waiting, Hoping to Weekly Buzz playlist"
    #sentence = "add red demon to my rock playlist"
    #sentence = 'book The Middle East restaurant in I.N. for noon'
    #sentence = "I\'d like to eat at a taverna that serves chili con carne for a party of 10."
    #sentence = "Add the current tune to my Rock Gaming playlist"
    #sentence = "Add villotta to The MetalSucks Playlist playlist"
    #sentence = "add Bronislau Kaper to my drive playlist"
    #sentence = "I want to call police for here right now"

    
    #sentence = "add Chris de Burgh in my playlist melancholia"
    #sentence = "Add John Tesh to my salsa classics playlist"
    #sentence = "add the current tune to my Six string peacefulness playlist"
    #sentence = "Add the track to my workout playlist."
    #sentence = "Add the album to my Heavy Gamer playlist."
    #sentence = "add an album in my playlist In The Name Of Blues"
    #sentence = "add bang a drum to my cleaning playlist"
    #sentence = "Add the track to the bandas sonoras playlist."
    #sentence = "Add the track to my romantic evening playlist."
    #sentence = "Add decade in the sun best of stereophonics to laundry."
    #sentence = "I'd like to put qriii on songs to sing in the car playlist."
    #sentence = "add Dunja, du to my ejercicio playlist"
    #sentence = "I want to send music to ann's lists from SIR John Winston Ono Lennon, put it on her Infinite Indie Folk list"
    #sentence = "add Runaljod gap var Ginnunga to ashlee's playlist titled forever country"
    #sentence = "Please put maimi yajima's song onto Operación Bikini."
    #sentence = "add artist in my Top 100 Rock Tracks on Spotify"
    #sentence = "Add the artist Jurij Szewczuk to tisha's Soul Revived playlist."
    #sentence = "Add Spanish Harlem Incident to cleaning the house"
    #sentence = "Add Live from Aragon Ballroom to Trapeo"
    sentence = "add saxophone supremacy to ofelia's calm before the storm playlist"
    #sentence = "add The Greyest of Blue Skies in Indie Español my playlist"
    #sentence = "Add the name kids in the street to the plylist New Indie Mix"
    #sentence = "add the album radar latino"
    #sentence = "Add Tranquility to the Latin Pop Rising playlist."
    #sentence = "Add d flame to the Dcode2016 playlist."
    #sentence = "Add My Chemical Romance to my playlist Hard Rock"

    
    

    '''
    sentence = "How's the weather in Munchique National Natural Park"
    sentence = "Tell me the weather forecast for France"
    sentence = "Will there be wind in Hornitos DC?"
    sentence = "Is it warm here now?"
    sentence = "what is the forecast for Roulo for foggy conditions on February the eighteenth, 2018"
    sentence = "I want to eat in Wallis and Futuna and get a blizzard"
    sentence = "what is the forecast in 23 hours and seventeen seconds at this current spot"
    sentence = "I need a table at a restaurant in Montserrat"
    sentence = "What is the weather forecast for one am. in West Athens, Benin"
    sentence = "What will the weather be this Afternoon in Brilliant MI?"
    sentence = "What is the weather not far from Upper Klamath National Wildlife Refuge?"
    sentence = "Weather in Parc national de Nech Sar"
    sentence = "What will the weather be in Battle Lake?"
    sentence = "Will it be temperate in the current place?"
    sentence = "What is the weather for eight o'clock in Laos that is within walking distance"
    sentence = "Tell me if it will it be windy in Grizzled Squirrel Wildlife Sanctuary"
    sentence = "Will it be windy here on Dec. 16th?"
    #sentence = "What is the weather forecast for Danvers"
    #sentence = "Will it be freezing on Jul. 7, 2038 in Vananda Alaska?"
    #sentence = "What will the weather be in Sche-Phoksundo-Nationalpark on leif erikson day?"
    #sentence = "Is the weather chilly in Pintura"
    #sentence = "What's the weather in Nigeria?"'''
    #sentence = "What is the overcast forecast within the same area of Rio Grande Nature Center State Park for fifteen o'clock"

    #sentence = "what is the forecast for Roulo for foggy conditions on February the eighteenth, 2018"



    sentence = "Rate All That Remains a five"
    sentence = "Give this album 4 points"
    


    
    result = main(sentence)

Intent reconhecido: 2.189511 segundos
Token: Give, POS: VERB, Dependência: ROOT
Token: this, POS: DET, Dependência: det
Token: album, POS: NOUN, Dependência: dative
Token: 4, POS: NUM, Dependência: nummod
Token: points, POS: NOUN, Dependência: npadvmod
Ignorando VERB ROOT 'Give' para evitar capturar a sentença inteira.
Capturado NOUN/PROPN (não compound): 'album'
Capturado NOUN/PROPN (não compound): 'points'
nouns_and_tokens capturados: ['album', 'points']
Syntactic_slots: 
 ['album', 'points']
Texto: 4 | Tipo: CARDINAL
Entities: 
 ['4']
Recuperação concluída em 0.23 segundos.
['Give (album)', 'Give It To Me (album)', 'Give It Up (album)', 'Give It to Me (album)', 'Give Up (album)', 'Never Give In (album)', "Something's Gotta Give (album)", 'What You Give Is What You Get (album)', 'You Get What You Give (album)', 'Give (album)', 'Give It to Me (album)', 'Never Give In (album)', "Something's Gotta Give (album)", 'What You Give Is What You Get (album)', 'You Get What You Give (album)', '

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


slots check score:  {'rating value': 0.9869192242622375, 'rating unit': 0.9818013906478882, 'object name': 0.12992329895496368, 'object part of series type': 0.07838431000709534, 'object type': 0.043139901012182236, 'object select': 0.01642812415957451, 'best rating': 0.009472901932895184}
Token                              4  4 points     album    points
best rating                 0.007995  0.007408  0.008653  0.008395
object name                 0.087414  0.088695  0.129105  0.030226
object part of series type  0.038980  0.070722  0.078013  0.076713
object select               0.015922  0.016214  0.016392  0.016326
object type                 0.038365  0.041867  0.042422  0.042252
rating unit                 0.969819  0.975710  0.939899  0.977599
rating value                0.975602  0.981585  0.942344  0.959399
{'intent': ('Rate an item', 0.996), 'slots': {'object part of series type': ('NULL', 0.0), 'object select': ('NULL', 0.0), 'rating unit': ('points', 0.978), 'rating value': 

In [11]:
def main(sentence):

    #start time tracker
    tracker.start()
    intent, intent_score = nlu_module.intent_recognition(sentence)
    tracker.mark("Intent reconhecido")

    if intent is None:
        print("Intent not recognized. Exiting...")
        return None
    slot_labels = list(nlu_module.intent_slot_dict[intent].get("slots", {}).keys())

    
    syntactic_slots = nlu_module.get_slot_tokens(sentence, slot_labels)
    print(f"Syntactic_slots: \n {syntactic_slots}")


    
    entities = nlu_module.get_entities(sentence)
    print(f"Entities: \n {entities}")
    


    
    query_results = engine.retrieve(sentence)
    doc_titles = query_results['title'].tolist()
    print(doc_titles)


    
    slots_retrieved = find_all_matching_substrings(doc_titles, sentence)
    slots_retrieved = nlu_module.filter_doctitles(slots_retrieved, slot_labels)
    print(f"Slots retrieved: \n {slots_retrieved}")



    
    relevant_tokens = nlu_module.remove_duplicates_and_substrings(syntactic_slots + slots_retrieved)
    print(f"Relevant tokens: \n {relevant_tokens}")
    tracker.mark("Tokens extraídos")


    qa_tokens = nlu_module.get_intent_slot_answers(intent, sentence)


    final_slot_tokens = nlu_module.remove_duplicates(relevant_tokens + entities + qa_tokens)
    final_slot_tokens = final_slot_tokens
    print(f"Final slot tokens: \n {final_slot_tokens}")



    
    df_slot_score = nlu_module.slot_scores(sentence, intent, final_slot_tokens)
    tracker.mark("Matriz de score slot-token concluida")

    # Remoção de redundancia:

    new_final_slot_tokens = process_until_no_more_groups(df_slot_score)

    final_df_slot_score = nlu_module.slot_scores(sentence, intent, new_final_slot_tokens)

    

    


    # NORMAL HUNGARIAN

    print("\n\n NORMAL HUNGARIAN \n\n ###################################\n\n")

    slots = nlu_module.slot_filling(final_df_slot_score)
    tracker.mark("Hungarian algoritm concluido")

    result = {'intent': (intent, intent_score), 'slots': slots}
    print(result)


    # MAX ASSOCIATIONS SEM REDUNDANCIA DE TOKENS PÓS HUNGARIAN

    print("\n\n MAX ASSOCIATIONS SEM REDUNDANCIA DE TOKENS PÓS HUNGARIAN \n\n ###################################\n\n")


    new_validation = select_max_triples(final_df_slot_score)
    print(new_validation)



    return df_slot_score

In [1]:
import pandas as pd

def normalize_text_span(text):
    """
    Função de normalização que apenas faz split case-sensitive,
    sem remover pontuação. Ajuste se quiser ignorar apóstrofos etc.
    """
    return text.split()

def share_word(token_a, token_b):
    """
    Retorna True se token_a e token_b compartilham ao menos uma palavra.
    Caso contrário, False.
    """
    words_a = set(normalize_text_span(token_a))
    words_b = set(normalize_text_span(token_b))
    return len(words_a & words_b) > 0

def can_form_groups(items):
    """
    Recebe uma lista de tuplas (span, score).
    1) Ordena descrescente por score.
    2) Tenta formar grupos sem 'cadeia transitiva':
       - cada novo span deve compartilhar palavras com TODOS do grupo.
    3) Se existir ao menos um grupo com 2 ou mais spans, retorna True.
       Caso contrário, False.
    """
    # Ordenar desc por score
    items_sorted = sorted(items, key=lambda x: x[1], reverse=True)
    
    # Formar grupos
    grupos = []
    for span, score in items_sorted:
        colocado = False
        for grupo in grupos:
            if all(share_word(span, g[0]) for g in grupo):
                grupo.append((span, score))
                colocado = True
                break
        if not colocado:
            grupos.append([(span, score)])
    
    # Verificar se existe um grupo com 2 ou mais spans
    for grupo in grupos:
        if len(grupo) >= 2:
            return True
    return False

def form_groups(items):
    """
    Recebe (span, score), ordena desc por score e forma grupos
    onde cada novo span deve compartilhar palavras com TODOS os do grupo.
    
    Retorna a lista de grupos, 
    onde cada grupo é uma lista de tuplas (span, score).
    """
    items_sorted = sorted(items, key=lambda x: x[1], reverse=True)
    grupos = []
    
    for span, score in items_sorted:
        colocado = False
        for grupo in grupos:
            if all(share_word(span, g[0]) for g in grupo):
                grupo.append((span, score))
                colocado = True
                break
        if not colocado:
            grupos.append([(span, score)])
    return grupos


def process_until_no_more_groups(df, threshold=0.2):
    """
    Algoritmo principal:
      Enquanto for possível formar ao menos um grupo >= 2 (can_form_groups == True):
        - form_groups(items)
        - printar os grupos
        - selecionar o melhor span em cada grupo
        - items = lista desses spans escolhidos
      Se não for possível formar grupo >= 2, encerra.
    """

    print("DataFrame original:\n", df, "\n")

    # -----------------------------
    # PASSO 1: Filtra colunas por threshold
    # -----------------------------
    keep_cols = [col for col in df.columns if df[col].max() > threshold]
    df_filtered = df[keep_cols]
    print(f"[PASSO 1] Tokens com score máximo > {threshold}: {list(df_filtered.columns)}")
    if df_filtered.empty:
        print("Nenhum token atende ao threshold. Encerrando.\n")
        return []
        
    print("DataFrame após filtragem:\n", df_filtered, "\n")

    # Lista de tokens restantes
    tokens_restantes = list(df_filtered.columns)

    # -----------------------------
    # PASSO 2: Criar lista de (span, total_score)
    # -----------------------------
    items = []
    for span in df_filtered.columns:
        total_score = df_filtered[span].sum()
        items.append((span, total_score))


    
    round_num = 1
    while True:
        print(f"\n=== ROUND {round_num} ===")
        can_group = can_form_groups(items)
        if not can_group:
            print("Não é possível formar nenhum grupo com 2 ou mais spans.")
            break

        # Se for possível formar grupos >= 2, formamos de fato
        grupos = form_groups(items)

        print("\nGrupos formados:")
        for i, grupo in enumerate(grupos, start=1):
            print(f"  Grupo {i}: {grupo}")

        final_spans = []
        print("\n[PASSO 5] Selecionando 1 span por grupo (o de maior score):")
        for i, grupo in enumerate(grupos, start=1):
            best_tuple = max(grupo, key=lambda x: x[1])  # x[1] => score
            final_spans.append(best_tuple)
            print(f"  Grupo {i}: {grupo} => Escolhido: {best_tuple}")

        print("\n[FINAL] Lista de tuplas escolhidas (um por grupo):", final_spans)

        # Atualiza 'items' para uma nova iteração
        items = final_spans
        round_num += 1

    # Retorna o estado final
    print("\n[ENCERRADO] Estado final de items:", items)
    spans = [span for span, score in items]

    
    return spans

#chosen_tuples = process_until_no_more_groups(result, threshold=0.2)
#print("Resultado final (tuplas):", chosen_tuples)

In [341]:
#nlu_module.validate_slot_assignments(sentence = sentence, result = result )

# PROBLEMAS ATUAIS

# MELHORAR A REMOÇÃO DE CHUNKS COM MESMOS TOKENS (PALAVRA SEGUIDA POR VIRGULA PASSA BATIDO PELO STRIP() )

# REMOVER CHUNKS QUE SEJAM IGUAIS A SENTENÇA INTEIRA

# VERIFICAR IMPACT DE MULTI_LABEL SER TRUE OU FALSE

# VERIFICAR SE ALGORITMO HUNGARO É A MELHOR ESCOLHA PARA ASSOCIAR CHUNKS COM TOKENS


# 30/03 - Colocar threshold inferior na escolha dos scores pelo hungarian algorithm

{'intent': ('Add to a playlist', 0.997),
 'slots': {'artist name': ('NULL', 0.0, 0.0),
  'music title': ('the current tune', 0.891),
  'playlist owner': ('my Rock Gaming', 0.868),
  'playlist title': ('Rock Gaming', 0.972)}}

In [ ]:
def remove_token_intersection(df, threshold=0.2):

    print("DataFrame original:\n", df, "\n")

    # -----------------------------
    # PASSO 1: Filtra colunas por threshold
    # -----------------------------
    keep_cols = [col for col in df.columns if df[col].max() > threshold]
    df_filtered = df[keep_cols]
    print(f"[PASSO 1] Tokens com score máximo > {threshold}: {list(df_filtered.columns)}")
    if df_filtered.empty:
        print("Nenhum token atende ao threshold. Encerrando.\n")
        return []
        
    print("DataFrame após filtragem:\n", df_filtered, "\n")

    # Lista de tokens restantes
    tokens_restantes = list(df_filtered.columns)

    # -----------------------------
    # PASSO 2: Criar lista de (span, total_score)
    # -----------------------------
    items = []
    for span in df_filtered.columns:
        total_score = df_filtered[span].sum()
        items.append((span, total_score))




    while True:
        can_form_groups(items)
    
    # -----------------------------
    # PASSO 3: Ordenar descrescente por total_score
    # -----------------------------
    items_sorted = sorted(items, key=lambda x: x[1], reverse=True)
    print("[PASSO 3] Lista de spans ordenada por score (desc):")
    for span, score in items_sorted:
        print(f"  - {span}: score total = {score:.4f}")
    print()

    
    # -----------------------------
    # PASSO 4: Formar grupos
    #     Critério: para cada (span,score), tenta encaixar num grupo existente
    #     se ele compartilha palavras com TODOS do grupo.
    # -----------------------------
    grupos = []
    print("[PASSO 4] Formando grupos sem 'cadeia transitiva'...")

    for span, score in items_sorted:
        print(f"\nAnalisando span '{span}' (score={score:.4f})")
        colocado = False
        for i, grupo in enumerate(grupos, start=1):
            # Verifica se esse 'span' casa com todos do grupo
            if all(share_word(span, g[0]) for g in grupo):
                print(f"  -> Entra no Grupo {i}, pois compartilha palavras com todos do grupo.")
                grupo.append((span, score))
                colocado = True
                break
            else:
                print(f"  -> Não consegue entrar no Grupo {i}. Tentando próximo grupo...")

        if not colocado:
            # Cria um novo grupo com esse span
            print(f"  -> Criando novo grupo para '{span}'")
            grupos.append([(span, score)])

    # Exibe os grupos resultantes
    print("\nGrupos formados:")
    for i, grupo in enumerate(grupos, start=1):
        print(f"  Grupo {i}: {grupo}")


    # -----------------------------
    # PASSO 5: Selecionar 1 tupla por grupo
    # Se a ideia é escolher o maior score do grupo, 
    # mas a gente já insere do maior para o menor,
    # no final, 'Grupo i' pode ter mais de um span.
    # Mantemos o que tiver maior score? 
    # Ou mantemos todos, se "máximo de grupos"?
    #
    # Abaixo escolhemos só 1 (o maior), 
    # mas já que estamos inserindo do maior pro menor,
    # na prática o primeiro será o maior do grupo.
    # Vou demonstrar a lógica de "pegar o maior" no final de qualquer forma.
    # -----------------------------
    final_spans = []
    print("\n[PASSO 5] Selecionando 1 span por grupo (o de maior score):")

    for i, grupo in enumerate(grupos, start=1):
        # 'grupo' é lista de tuplas (span, score)
        best_tuple = max(grupo, key=lambda x: x[1])  # x[1] => score
        final_spans.append(best_tuple)
        print(f"  Grupo {i}: {grupo} => Escolhido: {best_tuple}")

    print("\n[FINAL] Lista de tuplas escolhidas (um por grupo):", final_spans)
    print("=== FIM DO PROCESSAMENTO ===\n")

    return final_spans
        




chosen_tuples = remove_token_intersection(result, threshold=0.2)
print("Resultado final (tuplas):", chosen_tuples)
    

DataFrame original:
 Token                              4  4 points     album    points
Slot                                                              
best rating                 0.843995  0.781988  0.913396  0.886195
object name                 0.672811  0.682670  0.993698  0.232642
object part of series type  0.497295  0.902241  0.995265  0.978678
object select               0.969184  0.986959  0.997806  0.993780
object type                 0.889305  0.970486  0.983348  0.979423
rating unit                 0.987796  0.993796  0.957320  0.995720
rating value                0.988533  0.994595  0.954834  0.972115 

[PASSO 1] Tokens com score máximo > 0.2: ['4', '4 points', 'album', 'points']
DataFrame após filtragem:
 Token                              4  4 points     album    points
Slot                                                              
best rating                 0.843995  0.781988  0.913396  0.886195
object name                 0.672811  0.682670  0.993698  0.232642
o

In [1]:
# INCLUIR ASPAS E HIFEN NOS TOKENS


import spacy

def fuse_special_tokens(doc):
    """
    Percorre os tokens do spaCy e funde manualmente os tokens especiais de pontuação,
    de forma que:
      - Se o token é um hífen ("-") e está classificado como 'PUNCT' ou 'PART',
        então funde o token anterior, o hífen e o token seguinte em um único token.
      - Para outros tokens especiais, como "'s" ou "’s", funde o token com o token anterior.
    
    Retorna:
      - Uma lista de strings, onde os tokens fundidos aparecem como um único token.
    """
    # Conjunto de símbolos que queremos fundir
    symbols_to_fuse = {"'s", "'", "-", "’s"}
    fused_tokens = []
    i = 0
    while i < len(doc):
        token = doc[i]
        # Se o token é especial e está na lista de símbolos:
        if token.pos_ in {"PART", "PUNCT"} and token.text in symbols_to_fuse:
            if token.text == "-":
                # Para hífen, juntamos o token anterior, o hífen e o token seguinte
                if fused_tokens and (i < len(doc) - 1):
                    # Concatena sem espaço: ex: "Jean-Luc" (já presente) + "-" + "s" (do próximo token)
                    fused = fused_tokens[-1] + token.text + doc[i+1].text
                    print(f"Fusão de hífen: '{fused_tokens[-1]}' + '{token.text}' + '{doc[i+1].text}' => '{fused}'")
                    fused_tokens[-1] = fused  # atualiza o token anterior com a fusão
                    i += 2  # pula o token do hífen e o token seguinte
                    continue
                else:
                    # Se não houver token anterior ou não houver token seguinte, adiciona normalmente.
                    fused_tokens.append(token.text)
                    i += 1
                    continue
            else:
                # Para outros símbolos (ex.: "'s", "'" ou "’s"): funde somente ao token anterior.
                if fused_tokens:
                    fused = fused_tokens[-1] + token.text
                    print(f"Fusão de símbolo: '{fused_tokens[-1]}' + '{token.text}' => '{fused}'")
                    fused_tokens[-1] = fused
                else:
                    fused_tokens.append(token.text)
                i += 1
                continue
        else:
            # Se token é normal, simplesmente adiciona.
            fused_tokens.append(token.text)
            i += 1
    return fused_tokens

# Exemplo de uso
if __name__ == "__main__":
    nlp = spacy.load("en_core_web_sm")
    
    # Exemplo de sentenças de teste
    test_sentences = [
        "I love rock'n'roll.",                           # deve manter "rock'n'roll" intacto
        "Jean-Luc's style is amazing.",                  # deve fundir "Jean-Luc", "-", "s" em "Jean-Luc's"
        "A well-known fact should be noted.",            # deve fundir "well-known" em um único token
        "He said, \"Hello, world!\"",                    # sem alteração nos tokens das aspas
        "I saw a high-speed train.",                     # deve fundir "high-speed" em um único token
    ]
    
    for sent in test_sentences:
        print("\n-----------------------------------------")
        print(f"Sentença: {sent}")
        doc = nlp(sent)
        print("Tokens originais:")
        for token in doc:
            print(f" - Token: '{token.text}', POS: {token.pos_}, Dep: {token.dep_}")
        
        fused = fuse_special_tokens(doc)
        print("Tokens após fusão:")
        for t in fused:
            print(f" - {t}")



-----------------------------------------
Sentença: I love rock'n'roll.
Tokens originais:
 - Token: 'I', POS: PRON, Dep: nsubj
 - Token: 'love', POS: VERB, Dep: ROOT
 - Token: 'rock'n'roll', POS: PROPN, Dep: dobj
 - Token: '.', POS: PUNCT, Dep: punct
Tokens após fusão:
 - I
 - love
 - rock'n'roll
 - .

-----------------------------------------
Sentença: Jean-Luc's style is amazing.
Tokens originais:
 - Token: 'Jean', POS: PROPN, Dep: compound
 - Token: '-', POS: PUNCT, Dep: punct
 - Token: 'Luc', POS: PROPN, Dep: poss
 - Token: ''s', POS: PART, Dep: case
 - Token: 'style', POS: NOUN, Dep: nsubj
 - Token: 'is', POS: AUX, Dep: ROOT
 - Token: 'amazing', POS: ADJ, Dep: acomp
 - Token: '.', POS: PUNCT, Dep: punct
Fusão de hífen: 'Jean' + '-' + 'Luc' => 'Jean-Luc'
Fusão de símbolo: 'Jean-Luc' + ''s' => 'Jean-Luc's'
Tokens após fusão:
 - Jean-Luc's
 - style
 - is
 - amazing
 - .

-----------------------------------------
Sentença: A well-known fact should be noted.
Tokens originais:
 - Token

In [ ]:
#sentence = "Help me find a restaurant in downtown São Paulo serving Brazilian cuisine for 4 people at Fogo de Chão, and please note the reservation time is 8:00 PM."
doc = nlu_module.nlp(sentence)
entities = [ent.text for ent in doc.ents]
print(entities)


for token in doc:
    print(f"\nToken: {token.text}, POS: {token.pos_}, Tag: {token.tag_}, Explanation: {spacy.explain(token.tag_)}")

def is_string_not_present(query, intents_dict):
    """
    Returns True if the query is not found as a substring in any intent or slot name.
    """
    query_lower = query.lower()
    for intent, data in intents_dict.items():
        if query_lower in intent.lower():
            return False
        slots = data.get("slots", {})
        for slot_name in slots.keys():
            if query_lower in slot_name.lower():
                return False
    return True

# Extracting nouns that are not substrings of any intent or slot names
filtered_nouns = [token.text for token in doc if (token.pos_ == "NOUN" or token.pos_ == "PROPN") and is_string_not_present(token.text, nlu_module.INTENT_SLOT_DICT)]

print(filtered_nouns)

In [125]:
nome_arquivo = '/home/felipe/thiago_project/files/Wiki_corpus_title_as_text.jsonl'

with open(nome_arquivo, 'r', encoding='utf-8') as f:
    linhas = f.readlines()

print(f"O arquivo '{nome_arquivo}' possui {len(linhas)} linhas.")


O arquivo '/home/felipe/thiago_project/files/Wiki_corpus_title_as_text.jsonl' possui 5848875 linhas.


In [ ]:
O arquivo '/home/felipe/thiago_project/files/enwiki-20250220-all-titles' possui 62603372 linhas.
O arquivo '/home/felipe/thiago_project/files/Wiki_corpus_title_as_text.jsonl' possui 5848875 linhas.


In [172]:
# Exemplo com uma expressão composta
text = "Add clem burke in my playlist Pre-Party R&B Jams"
doc = nlu_module.nlp(text)

# Verifica entidades nomeadas
for ent in doc.ents:
    print(f"Entidade: {ent.text} - Tipo: {ent.label_}")

# Exibe informações sintáticas de cada token
for token in doc:
    print(f"Token: {token.text}, POS: {token.pos_}, Dependência: {token.dep_}")

Entidade: Pre-Party R&B Jams - Tipo: ORG
Token: Add, POS: VERB, Dependência: ROOT
Token: clem, POS: NOUN, Dependência: compound
Token: burke, POS: NOUN, Dependência: dobj
Token: in, POS: ADP, Dependência: prep
Token: my, POS: PRON, Dependência: poss
Token: playlist, POS: NOUN, Dependência: pobj
Token: Pre, POS: PROPN, Dependência: compound
Token: -, POS: PROPN, Dependência: compound
Token: Party, POS: PROPN, Dependência: compound
Token: R&B, POS: PROPN, Dependência: compound
Token: Jams, POS: PROPN, Dependência: appos


In [419]:
import stanza

# Baixar o modelo para inglês (somente na primeira vez)
stanza.download('en')

# Criar o pipeline de NLP do Stanza (inclui tokenização, POS, dependências)
nlp = stanza.Pipeline('en', processors='tokenize,pos,lemma,depparse')

# Sentença de exemplo
sentence = "add crying, waiting, hoping to weekly buzz playlist."

# Processar o texto
doc = nlp(sentence)

# Exibir os tokens e suas dependências
print(f"{'Token':<10} {'POS':<10} {'Dependência':<15} {'Governador':<10}")
print("="*50)
for sentence in doc.sentences:
    for word in sentence.words:
        print(f"{word.text:<10} {word.upos:<10} {word.deprel:<15} {sentence.words[word.head-1].text if word.head > 0 else 'ROOT'}")


2025-03-14 05:29:16 INFO: Downloaded file to /home/felipe/stanza_resources/resources.json
2025-03-14 05:29:16 INFO: Downloading default packages for language: en (English) ...


2025-03-14 05:29:28 INFO: Downloaded file to /home/felipe/stanza_resources/en/default.zip
2025-03-14 05:29:31 INFO: Finished downloading models and saved to /home/felipe/stanza_resources
2025-03-14 05:29:31 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-03-14 05:29:31 INFO: Downloaded file to /home/felipe/stanza_resources/resources.json
2025-03-14 05:29:31 WARNING: Language en package default expects mwt, which has been added
2025-03-14 05:29:32 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

2025-03-14 05:29:32 INFO: Using device: cuda
2025-03-14 05:29:32 INFO: Loading: tokenize
2025-03-14 05:29:32 INFO: Loading: mwt
2025-03-14 05:29:32 INFO: Loading: pos
2025-03-14 05:29:34 INFO: Loading: lemma
2025-03-14 05:29:34 INFO: Loading: depparse
2025-03-14 05:29:34 INFO: Done loading processors!


Token      POS        Dependência     Governador
add        VERB       root            ROOT
crying     NOUN       obj             add
,          PUNCT      punct           waiting
waiting    NOUN       conj            add
,          PUNCT      punct           hoping
hoping     VERB       advcl           add
to         PART       case            playlist
weekly     ADJ        amod            playlist
buzz       NOUN       compound        playlist
playlist   NOUN       obj             hoping
.          PUNCT      punct           add
